# 5. Cải tiến: kết hợp mô hình & tổng hợp thực nghiệm

**Input:** `exps/feature0/preds/*_oof.npy, *_test.npy` (do stage 4 sinh ra)  
**Output:** `submission_blend.csv`, `experiments.xlsx`

BEGIN

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys; sys.path.insert(0, "../../extra")
from config import *
from common import *
from utils import *
display.clear_output()

In [3]:
MEMBER   = "TV4"      # tên/mã thành viên -> file log riêng
NOTEBOOK = "improve"
FEATURE_SET = "feature5"
seed_everything()

In [4]:
from scipy.optimize import nnls
x_train, y_train, x_test, test_id = load_feature_set(FEATURE_SET)
d = f'{feature_dir(FEATURE_SET)}/preds'
names = sorted(f[:-8] for f in os.listdir(d) if f.endswith('_oof.npy'))
OOF = np.column_stack([np.load(f'{d}/{n}_oof.npy') for n in names])
TEST = np.column_stack([np.load(f'{d}/{n}_test.npy') for n in names])
pd.Series({n: rmse(y_train, OOF[:, i]) for i, n in enumerate(names)}).sort_values().to_frame('OOF RMSE')

[load] feature5: x_train (1456, 262), x_test (1459, 262)


,OOF RMSE
TV2_Lasso_tuned,0.111993
TV2_GBR_tuned,0.112294
TV2_Ridge_tuned,0.115148
TV2_SVR,0.116226
TV2_ENet_tuned,0.116304
TV2_ENet,0.117438
TV2_Lasso,0.117469
TV3_final,0.117780
TV2_GBR,0.119906
TV2_Ridge,0.122168


## 5.1. Blend: trọng số không âm (NNLS) trên dự đoán out-of-fold
Trọng số chọn trên chính OOF nên điểm hơi lạc quan (số tham số nhỏ nên lạc quan ít).

In [5]:
w, _ = nnls(OOF, y_train); w = w / w.sum()
print({n: round(float(v), 3) for n, v in zip(names, w) if v > 0.001})
blend_oof = OOF @ w
scores = np.array([rmse(y_train[va], blend_oof[va]) for _, va in get_kfold().split(x_train)])
log_experiment(MEMBER, NOTEBOOK, FEATURE_SET, 'Blend', scores, {n: round(float(v), 3) for n, v in zip(names, w)})
save_submission(FEATURE_SET, 'blend', test_id, TEST @ w)

{'TV2_GBR_tuned': 0.41, 'TV2_Lasso_tuned': 0.247, 'TV2_SVR': 0.181, 'TV3_final': 0.161}
[log] Blend        | feature5/all | CV RMSE = 0.10740 ± 0.00766
[submit] d:/HocSau/House_Prices/house_prices_project/exps/feature5\submission_blend.csv


'd:/HocSau/House_Prices/house_prices_project/exps/feature5\\submission_blend.csv'

In [6]:
for n in ['TV2_Lasso_tuned', 'TV2_GBR_tuned', 'TV3_final']:
    save_submission(FEATURE_SET, n, test_id, np.load(f'{d}/{n}_test.npy'))

[submit] d:/HocSau/House_Prices/house_prices_project/exps/feature5\submission_TV2_Lasso_tuned.csv
[submit] d:/HocSau/House_Prices/house_prices_project/exps/feature5\submission_TV2_GBR_tuned.csv
[submit] d:/HocSau/House_Prices/house_prices_project/exps/feature5\submission_TV3_final.csv


## 5.2. Tổng hợp log của cả nhóm → `experiments.xlsx`

In [7]:
export_experiments().head(20)

[export] 99 thực nghiệm -> d:/HocSau/House_Prices/house_prices_project/experiments.xlsx


,No,time,member,notebook,feature_set,selection,model,params,cv_rmse,cv_std,kaggle_score,note
0,1,2026-10-05 15:47,TV4,improve,feature5,all,Blend,"{'TV2_ENet': 0.0, 'TV2_ENet_tuned': 0.0, 'TV2_...",0.10740,0.00766,0.12826,NaN
1,2,2026-10-05 14:25,TV4,feature_selection4,feature4,gbr_top50,Ridge,NaN,0.11141,0.00587,NaN,NaN
2,3,2026-10-05 15:23,TV2,traditional model2,feature5,all,Lasso_tuned,{'model__alpha': 0.003},0.11179,0.00676,0.13164,NaN
3,4,2026-10-05 15:23,TV2,traditional model2,feature5,all,GBR_tuned,"{'model__max_depth': 2, 'model__max_features':...",0.11210,0.00652,0.12805,NaN
4,5,2026-10-05 15:23,TV2,traditional model2,feature5,all,Ridge_tuned,{'model__alpha': 300},0.11491,0.00729,NaN,NaN
5,6,2026-10-05 15:23,TV2,traditional model2,feature5,all,SVR,"{'C': 20, 'cache_size': 200, 'coef0': 0.0, 'de...",0.11593,0.00828,NaN,NaN
6,7,2026-10-05 15:23,TV2,traditional model2,feature5,all,ENet_tuned,"{'model__alpha': 0.001, 'model__l1_ratio': 0.7}",0.11616,0.00568,NaN,NaN
7,8,2026-10-05 15:27,TV3,neural network model3,feature5,all,ES_drop0.3_h512-256,"{'hidden': (512, 256), 'dropout': 0.3, 'lr': 0...",0.11642,0.01142,NaN,NaN
8,9,2026-10-05 14:25,TV4,feature_selection4,feature4,lasso,Ridge,NaN,0.11661,0.00492,NaN,NaN
9,10,2026-10-05 15:27,TV3,neural network model3,feature5,all,ES_drop0.3_BN,"{'hidden': (256, 128), 'dropout': 0.3, 'lr': 0...",0.11687,0.01022,NaN,NaN


In [8]:
def set_kaggle(member, model, feature_set, score):
    """Ghi điểm Kaggle vào dòng MỚI NHẤT của (model, feature_set) trong log của member."""
    path = f'{logs_dir}/experiments_{member}.csv'
    d = pd.read_csv(path)
    idx = d[(d.model == model) & (d.feature_set == feature_set)].index[-1]
    d.loc[idx, 'kaggle_score'] = score
    d.to_csv(path, index=False)
    print(f'{member} | {model} | {feature_set} -> kaggle_score = {score}')

# Sửa số theo điểm Public Score bạn thấy trên Kaggle:
set_kaggle('TV4', 'Blend',       'feature5', 0.12826)
set_kaggle('TV2', 'Lasso_tuned', 'feature5', 0.13164)
set_kaggle('TV2', 'GBR_tuned',   'feature5', 0.12805)
set_kaggle('TV3', 'final',       'feature5', 0.13564)
export_experiments().head(10)

TV4 | Blend | feature5 -> kaggle_score = 0.12826
TV2 | Lasso_tuned | feature5 -> kaggle_score = 0.13164
TV2 | GBR_tuned | feature5 -> kaggle_score = 0.12805
TV3 | final | feature5 -> kaggle_score = 0.13564
[export] 99 thực nghiệm -> d:/HocSau/House_Prices/house_prices_project/experiments.xlsx


,No,time,member,notebook,feature_set,selection,model,params,cv_rmse,cv_std,kaggle_score,note
0,1,2026-10-05 15:51,TV4,improve,feature5,all,Blend,"{'TV2_ENet': 0.0, 'TV2_ENet_tuned': 0.0, 'TV2_...",0.10740,0.00766,0.12826,NaN
1,2,2026-10-05 14:25,TV4,feature_selection4,feature4,gbr_top50,Ridge,NaN,0.11141,0.00587,NaN,NaN
2,3,2026-10-05 15:23,TV2,traditional model2,feature5,all,Lasso_tuned,{'model__alpha': 0.003},0.11179,0.00676,0.13164,NaN
3,4,2026-10-05 15:23,TV2,traditional model2,feature5,all,GBR_tuned,"{'model__max_depth': 2, 'model__max_features':...",0.11210,0.00652,0.12805,NaN
4,5,2026-10-05 15:23,TV2,traditional model2,feature5,all,Ridge_tuned,{'model__alpha': 300},0.11491,0.00729,NaN,NaN
5,6,2026-10-05 15:23,TV2,traditional model2,feature5,all,SVR,"{'C': 20, 'cache_size': 200, 'coef0': 0.0, 'de...",0.11593,0.00828,NaN,NaN
6,7,2026-10-05 15:23,TV2,traditional model2,feature5,all,ENet_tuned,"{'model__alpha': 0.001, 'model__l1_ratio': 0.7}",0.11616,0.00568,NaN,NaN
7,8,2026-10-05 15:27,TV3,neural network model3,feature5,all,ES_drop0.3_h512-256,"{'hidden': (512, 256), 'dropout': 0.3, 'lr': 0...",0.11642,0.01142,NaN,NaN
8,9,2026-10-05 14:25,TV4,feature_selection4,feature4,lasso,Ridge,NaN,0.11661,0.00492,NaN,NaN
9,10,2026-10-05 15:27,TV3,neural network model3,feature5,all,ES_drop0.3_BN,"{'hidden': (256, 128), 'dropout': 0.3, 'lr': 0...",0.11687,0.01022,NaN,NaN


### Kết luận
- Blend 4 mô hình khác họ đạt 0.1074, tốt hơn mô hình đơn tốt nhất khoảng 0.004.